# Nombre: Byron Ortiz

# Web Scraping Exercise

## 1. Introduction and Planning

### Objective:
The goal of this exercise is to build a web scraper that collects data from a chosen website. You will learn how to send HTTP requests, parse HTML content, extract relevant data, and store it in a structured format.

### Tasks:
1. Identify the data you want to scrape.
2. Choose the target website(s).
3. Plan the structure of your project.

### Example:
For this exercise, we will scrape job listings from Indeed.com. We will extract job titles, company names, locations, and job descriptions.

## 2. Understanding the Target Website
### Objective:

Analyze the structure of the web pages to be scraped.
### Tasks:

* Inspect the target website using browser developer tools.
* Identify the HTML elements that contain the desired data.

### Instructions:

* Open your browser and navigate to the target website (e.g., Indeed.com).
* Right-click on the webpage and select "Inspect" or press Ctrl+Shift+I.
* Use the developer tools to explore the HTML structure of the webpage.
* Identify the tags and classes of the elements that contain the job titles, company names, locations, and descriptions.

## 3. Writing the Scraper
### Objective:

Develop the code to scrape data from the target website.
### Tasks:

* Send HTTP requests to the target website.
* Parse the HTML content and extract the required data.
* Handle pagination to scrape data from multiple pages.
* Implement error handling.

Se importa las siguientes librerias:

In [1]:
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm
import pandas as pd
import time

Se descarga el html de la pagina https://www.allrecipes.com/recipes-a-z-6735880 donde se encuentran los links de todas las recetas

Se uso cmd para su descarga: curl -o recetas.html https://www.allrecipes.com/recipes-a-z-6735880

Se carga el html descargado para obtener todos los links de la pagina

In [2]:
# Load the HTML file
with open("datos/recetas.html", "r", encoding="utf-8") as file:
    html_content = file.read()
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

Se obtiene los todos los enlaces de cada receta que existe en la pagina usando la etiqueta 'a' y sus respectivas clases

In [3]:
# Encontrar todos los enlaces
headers = {
    "User-Agent":"My Phyton App"
}
links=soup.find('div' , class_='comp mntl-alphabetical-list mntl-block').find_all('a')
total_links=[]
inicio = time.time()
for link in links:
    response = requests.get(link.get('href'), headers=headers)
    soup2 = BeautifulSoup(response.content, 'html.parser')
    recipe_cards = soup2.find_all('a', class_='comp mntl-card-list-items mntl-document-card mntl-card card card--no-image')
    urls = [link.get('href') for link in recipe_cards]
    total_links.extend(urls)
fin = time.time()

print(f'Tiempo total de procesamiento: {fin - inicio:.2f} segundos')
print(pd.DataFrame(total_links).head())
print(f'Número de recetas: {len(total_links)}')

Tiempo total de procesamiento: 318.40 segundos
                                                   0
0  https://www.allrecipes.com/air-fryer-buffalo-w...
1  https://www.allrecipes.com/air-fryer-smashed-p...
2  https://www.allrecipes.com/air-fryer-quesadill...
3  https://www.allrecipes.com/air-fryer-truffle-p...
4  https://www.allrecipes.com/air-fryer-firecrack...
Número de recetas: 18122


Se obtiene el titulo, descripcion, ingredientes, instrucciones e informacion nutricional de una receta en especifica usando el link de la receta

In [4]:
response = requests.get(total_links[0], headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]

# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]
# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Titulo de la receta:", title)
print("\nDescripcion:", description)
print("\nIngredientes:")
for ingredient in ingredients:
    print("-", ingredient)
print("\nInstrucciones:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("\nInformación nutricional:")
for fact in nutrition_facts:
    print("-", fact)

Titulo de la receta: Air Fryer Buffalo Wings

Descripcion: These air fryer Buffalo wings are perfectly seasoned and incredibly easy to prep in minutes for crispy, juicy, spicy wings without the mess!

Ingredientes:
- 2 teaspoons sea salt
- 1 teaspoon garlic powder
- 1 teaspoon mustard powder
- 1 teaspoon ground coriander
- 1 teaspoon smoked paprika
- ½ teaspoon cayenne pepper
- ¼ teaspoon freshly ground black pepper
- 1 pound chicken wings

Instrucciones:
1. Preheat an air fryer to 380 degrees F (190 degrees C).
2. Combine sea salt, garlic powder, mustard powder, coriander, smoked paprika, cayenne pepper, and black pepper in a shallow bowl. Dredge each chicken wing in the spice mixture and set into the air fryer basket in one layer.
3. Cook in the air fryer until chicken wings are no longer pink at the bone and the juices run clear, about 30 minutes. An instant-read thermometer inserted near the bone should read 165 degrees F (74 degrees C). If wings are smaller, reduce cooking time.



Una vez vemos funciona para una receta se procede hacerlo para mas recetas primero para 10 luego para 100 y despues de ver que no hubo ningun problema para varias recetas lo hacemos para todas las recetas

In [5]:
def fetch_recipe_content(url):
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors
        soup = BeautifulSoup(response.content, "html.parser")
        
        title = soup.find("meta", {"property": "og:title"})["content"]

        # Extracting the description
        description = soup.find("meta", {"name": "description"})["content"]
        # Extracting the ingredients
        ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
        ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

        # Extracting the instructions
        instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
        instructions = [instruction.get_text().strip() for instruction in instructions_section]

        # Extracting the nutrition information
        nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
        nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

        return True,[title, description, ingredients, instructions,nutrition_facts]
        
    except requests.exceptions.RequestException as e:
        return False,[]

en este caso solo se hizo para 100 links debido al timpo que tarda en realizar esta operacion.

In [6]:
# Loop through the list of recipe URLs and fetch their contents
titles = []
descriptions = []
ingredients_list = []
instructions_list = []
nutrition_list = []
for url in total_links[:100]:    
    bool,resultado=fetch_recipe_content(url)
    if bool:
        titles.append(resultado[0])
        descriptions.append(resultado[1])
        ingredients_list.append(resultado[2])
        instructions_list.append(resultado[3])
        nutrition_list.append(resultado[4])

Se crea un dataframe que contenga todos los datos guardados anteriormente para poder visualizarlo de una mejor manera

In [7]:
recetas_df=pd.DataFrame({'titulo':titles,'descripcion':descriptions,'ingredientes':ingredients_list,'instrucciones':instructions_list,'nutricion':nutrition_list})
recetas_df

,titulo,descripcion,ingredientes,instrucciones,nutricion
0,Air Fryer Buffalo Wings,These air fryer Buffalo wings are perfectly se...,"[2 teaspoons sea salt, 1 teaspoon garlic powde...",[Preheat an air fryer to 380 degrees F (190 de...,"[Total Fat 28g, Saturated Fat 10g, Cholesterol..."
1,Air Fryer Smashed Potatoes,These air fryer smashed potatoes are golden an...,"[8 ounces baby gold potatoes, 1 tablespoon mel...",[Preheat an air fryer to 400 degrees F (200 de...,"[Total Fat 6g, Saturated Fat 4g, Cholesterol 1..."
2,Air Fryer Quesadillas,These air fryer quesadillas are golden and cri...,"[2 flour tortillas, 1/2 cup shredded cheese, n...",[Heat tortillas in the microwave until pliable...,"[Total Fat 12g, Saturated Fat 6g, Cholesterol ..."
3,Air Fryer Truffle Polenta Fries,"These air fryer truffle polenta fries, flavore...","[1 (18 ounce) tube prepared polenta, 1 1/2 tab...",[Preheat an air fryer to 400 degrees F (200 de...,"[Total Fat 8g, Saturated Fat 3g, Cholesterol 1..."
4,Air Fryer Firecracker Salmon Bites,These air fryer firecracker salmon bites get a...,"[1/4 cup balsamic vinegar, 1/4 cup brown sugar...","[Combine balsamic vinegar, brown sugar, oil, s...","[Total Fat 21g, Saturated Fat 4g, Cholesterol ..."
...,...,...,...,...,...
95,Granola Cups,"These granola cups, filled with yogurt and fru...","[1/4 cup unsalted butter, 1/2 cup pure maple s...",[Place butter and maple syrup in a large bowl ...,"[Total Fat 9g, Saturated Fat 4g, Cholesterol 1..."
96,4-Ingredient Hamburger Casserole,This 4-ingredient hamburger casserole is so co...,"[8 ounces pasta, 1 pound ground beef, salt, fr...",[Preheat the oven to 350 degrees F (175 degree...,"[Total Fat 32g, Saturated Fat 14g, Cholesterol..."
97,Salmon Caesar Salad,"This salmon Caesar salad, with Romaine lettuce...","[1 (8 ounce) salmon filet, 2 teaspoons oil, 2 ...",[Set a nonstick skillet over medium high heat....,"[Total Fat 44g, Saturated Fat 9g, Cholesterol ..."
98,New York-Style Crumb Cake,New York-Style crumb cake is a tender cake kno...,"[4 cups cake flour, 1 cup brown sugar, 1/2 cup...",[Preheat the oven to 350 degrees F (175 degree...,"[Total Fat 23g, Saturated Fat 13g, Cholesterol..."


# Todas la recetas

para poder hacer web scrapping de todas las recetas se lo hizo en un script el mismo proceso pero para todo el corpus usando multiprocesamiento

Se importa los links de todas las recetas para poder usarlo en el script

In [8]:
import pickle
with open('datos/total_links.pkl', 'wb') as file:
    pickle.dump(total_links, file)

Se carca el archivo genereado por el script para poder visualizarlo aca

In [9]:
with open('corpus.pkl', 'rb') as file:
    corpus=pickle.load(file)

Se puede ver que se obtuvo la informacion mas relevante de todas las recetas con exito

In [10]:
corpus

,Titulo,Descripcion,Ingredientes,Instrucciones,Informacion Nutricional
0,Air Fryer Buffalo Wings,These air fryer Buffalo wings are perfectly se...,"[2 teaspoons sea salt, 1 teaspoon garlic powde...",[Preheat an air fryer to 380 degrees F (190 de...,"[Total Fat 28g, Saturated Fat 10g, Cholesterol..."
1,Air Fryer Smashed Potatoes,These air fryer smashed potatoes are golden an...,"[8 ounces baby gold potatoes, 1 tablespoon mel...",[Preheat an air fryer to 400 degrees F (200 de...,"[Total Fat 6g, Saturated Fat 4g, Cholesterol 1..."
2,Air Fryer Quesadillas,These air fryer quesadillas are golden and cri...,"[2 flour tortillas, 1/2 cup shredded cheese, n...",[Heat tortillas in the microwave until pliable...,"[Total Fat 12g, Saturated Fat 6g, Cholesterol ..."
3,Air Fryer Truffle Polenta Fries,"These air fryer truffle polenta fries, flavore...","[1 (18 ounce) tube prepared polenta, 1 1/2 tab...",[Preheat an air fryer to 400 degrees F (200 de...,"[Total Fat 8g, Saturated Fat 3g, Cholesterol 1..."
4,Air Fryer Firecracker Salmon Bites,These air fryer firecracker salmon bites get a...,"[1/4 cup balsamic vinegar, 1/4 cup brown sugar...","[Combine balsamic vinegar, brown sugar, oil, s...","[Total Fat 21g, Saturated Fat 4g, Cholesterol ..."
...,...,...,...,...,...
18117,Vegan Zucchini Banana Bread,This moist zucchini banana bread is dairy-free...,"[3 cups all-purpose flour, 1 teaspoon salt, 1 ...",[Preheat the oven to 325 degrees F (165 degree...,"[Total Fat 10g, Saturated Fat 1g, Sodium 389mg..."
18118,Zucchini-Raspberry Bread,A simple zucchini nut bread with an unexpected...,"[1 ½ cups self-rising flour, 1 teaspoon ground...",[Preheat an oven to 350 degrees F (175 degrees...,"[Total Fat 8g, Saturated Fat 1g, Cholesterol 1..."
18119,Healthier Mom's Zucchini Bread,This moist and flavorful zucchini bread is mad...,"[1 ½ cups all-purpose flour, 1 ½ cups white wh...",[Preheat oven to 325 degrees F (165 degrees C)...,"[Total Fat 9g, Saturated Fat 1g, Cholesterol 2..."
18120,"Zucchini Bread, Pumpkin Style",Here's a flavorful zucchini bread with a pumpk...,"[3 medium zucchini, cut into chunks, 4 ¾ cups ...",[Preheat an oven to 350 degrees F (175 degrees...,"[Total Fat 10g, Saturated Fat 2g, Cholesterol ..."
